# 06 - Evaluation Metrics (Google Colab)
Run inference on validation dataset, compute JSON validity rate, ATS score distribution, and sample predictions.

In [ ]:
# Mount Google Drive and set working directory
from google.colab import drive
drive.mount('/content/drive')

import os
COLAB_ROOT = '/content/drive/MyDrive/colab'
os.chdir(COLAB_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

import yaml
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm

In [ ]:
# Load config and model
with open("configs/training_config.yaml", "r") as f:
    config = yaml.safe_load(f)

model_name = config["model_name"]
adapter_path = config["output_dir"]
use_4bit = config.get("use_4bit", True)

# Quantization
bnb_config = None
if use_4bit and torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

# Load tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
except Exception:
    model_name = config["fallback_model"]
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model
model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.float16,
    "device_map": "auto",
}
if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

print(f"Model loaded: {model_name} + LoRA from {adapter_path}")

In [ ]:
# Load validation dataset
with open(config["validation_dataset"], "r", encoding="utf-8") as f:
    val_data = json.load(f)

print(f"Validation samples: {len(val_data)}")

In [ ]:
# Helper functions
def extract_json(text):
    json_match = re.search(r'\{[\s\S]*\}', text)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None

def validate_ats_output(output):
    required = {"ats_score", "score_breakdown", "matched_skills",
                "missing_skills", "weak_bullets", "formatting_issues", "overall_feedback"}
    if not isinstance(output, dict):
        return False
    if not required.issubset(output.keys()):
        return False
    if not isinstance(output["ats_score"], (int, float)) or not (0 <= output["ats_score"] <= 100):
        return False
    return True

def generate_single(sample, max_new_tokens=1024):
    instruction = sample["instruction"]
    input_text = sample["input"]
    prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

print("Helper functions defined")

In [ ]:
# Run inference on validation dataset
print(f"Running inference on {len(val_data)} validation samples...")
print("This may take a while depending on your hardware.\n")

results = []
json_valid_count = 0
ats_valid_count = 0
predicted_scores = []
ground_truth_scores = []
predicted_skills = []
gt_skills = []

for i, sample in enumerate(tqdm(val_data)):
    # Generate prediction
    raw_output = generate_single(sample)
    parsed = extract_json(raw_output)

    # Ground truth
    gt = json.loads(sample["output"])

    # Track metrics
    is_valid_json = parsed is not None
    is_valid_ats = is_valid_json and validate_ats_output(parsed)

    if is_valid_json:
        json_valid_count += 1
    if is_valid_ats:
        ats_valid_count += 1
        predicted_scores.append(parsed["ats_score"])
        ground_truth_scores.append(gt["ats_score"])
        predicted_skills.append(set(parsed.get("missing_skills", [])))
        gt_skills.append(set(gt.get("missing_skills", [])))

    results.append({
        "index": i,
        "valid_json": is_valid_json,
        "valid_ats": is_valid_ats,
        "predicted": parsed,
        "ground_truth": gt,
    })

print(f"\nInference complete!")

In [ ]:
# Metric 1: JSON Validity Rate
total = len(val_data)
json_rate = json_valid_count / total * 100
ats_rate = ats_valid_count / total * 100

print("=" * 50)
print("EVALUATION METRICS")
print("=" * 50)

print(f"\n1. JSON Validity Rate")
print(f"   Valid JSON outputs: {json_valid_count}/{total} ({json_rate:.1f}%)")
print(f"   Valid ATS structure: {ats_valid_count}/{total} ({ats_rate:.1f}%)")

target = 95
status = "PASSED" if json_rate >= target else "BELOW TARGET"
print(f"   Target (>{target}%): {status}")

In [ ]:
# Metric 2: ATS Score Distribution
print(f"\n2. ATS Score Distribution")

if predicted_scores:
    print(f"   Predicted scores:")
    print(f"     Min: {min(predicted_scores)}")
    print(f"     Max: {max(predicted_scores)}")
    print(f"     Mean: {sum(predicted_scores)/len(predicted_scores):.1f}")

    print(f"   Ground truth scores:")
    print(f"     Min: {min(ground_truth_scores)}")
    print(f"     Max: {max(ground_truth_scores)}")
    print(f"     Mean: {sum(ground_truth_scores)/len(ground_truth_scores):.1f}")

    # Score difference
    diffs = [abs(p - g) for p, g in zip(predicted_scores, ground_truth_scores)]
    print(f"   Mean absolute difference: {sum(diffs)/len(diffs):.1f}")

    # Score bucket distribution
    print(f"\n   Predicted score distribution:")
    buckets = Counter()
    for s in predicted_scores:
        buckets[(s // 10) * 10] += 1
    for bucket in sorted(buckets.keys()):
        bar = "#" * buckets[bucket]
        print(f"     {bucket:3d}-{bucket+9:3d}: {bar} ({buckets[bucket]})")
else:
    print("   No valid predictions to analyze")

In [ ]:
# Metric 3: Missing Skill Accuracy
print(f"\n3. Missing Skill Accuracy")

if predicted_skills:
    total_overlap = 0
    total_gt = 0
    total_pred = 0

    for pred, gt in zip(predicted_skills, gt_skills):
        # Normalize to lowercase for comparison
        pred_lower = {s.lower() for s in pred}
        gt_lower = {s.lower() for s in gt}
        total_overlap += len(pred_lower & gt_lower)
        total_gt += len(gt_lower)
        total_pred += len(pred_lower)

    recall = total_overlap / total_gt * 100 if total_gt > 0 else 0
    precision = total_overlap / total_pred * 100 if total_pred > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    print(f"   Precision: {precision:.1f}%")
    print(f"   Recall: {recall:.1f}%")
    print(f"   F1 Score: {f1:.1f}%")
else:
    print("   No valid predictions to analyze")

In [ ]:
# Sample Predictions Comparison
print("\n" + "=" * 50)
print("SAMPLE PREDICTIONS")
print("=" * 50)

for i, r in enumerate(results[:5]):
    print(f"\n--- Sample {i+1} ---")
    gt = r["ground_truth"]
    pred = r["predicted"]

    print(f"Valid JSON: {r['valid_json']}")
    print(f"Valid ATS: {r['valid_ats']}")

    if r["valid_ats"] and pred:
        print(f"\n  Ground Truth Score: {gt['ats_score']}")
        print(f"  Predicted Score:    {pred['ats_score']}")
        print(f"  Difference:         {abs(gt['ats_score'] - pred['ats_score'])}")

        print(f"\n  GT Matched Skills:  {gt['matched_skills'][:3]}")
        print(f"  Pred Matched Skills: {pred['matched_skills'][:3]}")

        print(f"\n  GT Missing Skills:  {gt['missing_skills'][:3]}")
        print(f"  Pred Missing Skills: {pred['missing_skills'][:3]}")
    else:
        raw = str(pred)[:200] if pred else "None"
        print(f"  Prediction: {raw}")

In [ ]:
# Summary Report
print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"  Total validation samples: {total}")
print(f"  JSON validity rate: {json_rate:.1f}%")
print(f"  ATS structure rate: {ats_rate:.1f}%")

if predicted_scores:
    mean_diff = sum(abs(p-g) for p,g in zip(predicted_scores, ground_truth_scores)) / len(predicted_scores)
    print(f"  Mean score difference: {mean_diff:.1f}")

print(f"\n  Quality targets:")
print(f"    JSON validity >95%: {'PASS' if json_rate > 95 else 'FAIL'}")
print(f"    Consistent scoring: {'PASS' if predicted_scores and mean_diff < 20 else 'NEEDS IMPROVEMENT'}")
print(f"    Valid ATS >90%: {'PASS' if ats_rate > 90 else 'FAIL'}")